# kompyle, Knowledge Compilation into *klay* Circuits

> **Repository:** https://github.com/ML-KULeuven/kompyle

---
## Installation

```bash
# clone and build
git clone https://github.com/ML-KULeuven/kompyle.git
cd kompyle
podman-compose build
podman-compose up -d dev
podman-compose exec dev bash
python -m venv .venv
source .venv/bin/activate
pip install -e ".[dev]"
```

Or, if you are installing from a wheel:

```bash
pip install kompyle
```

> **Note:** installing from wheel doesn't contain most features.

In [1]:
import kompyle as p
import klay    as k

print("kompyle exports:", [s for s in dir(p) if not s.startswith("_")])

kompyle exports: ['Circuit', 'GatedFormula', 'NodePtr', 'SDNNFResult', 'SDNNFViolation', 'check_decomposability', 'check_sdnnf', 'check_smooth', 'compile_from_cnf_using_d4v2', 'compile_from_cnf_using_ganak', 'compile_from_cnf_using_ganakarjun', 'compile_from_cnf_using_sdd', 'compile_from_gates_file_using_d4v2', 'compile_from_gates_formula_using_d4v2', 'compile_from_sdd', 'pkompyle']


In [2]:
# Helper writing a CNF to a temporary file
import os
import tempfile
from typing import List


def write_cnf(n_vars: int, clauses: List[List[int]]) -> str:
    """Write a CNF formula to a temporary DIMACS file and return the path."""
    fd, path = tempfile.mkstemp(suffix=".cnf")
    with os.fdopen(fd, "w") as f:
        f.write(f"p cnf {n_vars} {len(clauses)}\n")
        for clause in clauses:
            f.write(" ".join(map(str, clause)) + " 0\n")
    return path


# example 1: XOR(x1, x2) with a free variable x3
# clauses: (x1 OR x2) AND (-x1 OR -x2)
xor_path = write_cnf(n_vars=3, clauses=[[1, 2], [-1, -2]])

print("DIMACS file contents:")
with open(xor_path) as f:
    print(f.read())

# example e: trivially UNSAT
# x1 AND -x1
unsat_path = write_cnf(n_vars=1, clauses=[[1], [-1]])

print("Paths created:", xor_path, unsat_path)

DIMACS file contents:
p cnf 3 2
1 2 0
-1 -2 0

Paths created: /tmp/tmp2qbi13bx.cnf /tmp/tmpdzpyzo4t.cnf


---
## 1. The `Circuit` Object

Multiple formulas can be compiled *into the same circuit*.

In [3]:
circuit = p.Circuit()

print(f"Empty circuit with nb_nodes: {circuit.nb_nodes()}")
print(f"Empty circuit with nb_root_nodes: {circuit.nb_root_nodes()}")

Empty circuit with nb_nodes: 0
Empty circuit with nb_root_nodes: 0


---
## 2. Compiling CNF with Different Backends

Each `compile_from_cnf_*` function takes a `Circuit` and a path to a DIMACS `.cnf` file and returns a `NodePtr` pointing to the root of the compiled sub-circuit.

### 2.1 Ganak

In [4]:
circuit = p.Circuit()

# compile the XOR formula using Ganak
root_xor = p.compile_from_cnf_using_ganak(circuit, xor_path)

# always call set_root after compiling to register the result
circuit.set_root(root_xor)

# prune dead nodes
circuit.remove_unused_nodes()

print(f"[Ganak] XOR circuit with nodes: {circuit.nb_nodes()}, roots: {circuit.nb_root_nodes()}")

[Ganak] XOR circuit with nodes: 15, roots: 1


### 2.2 Ganak + Arjun

In [5]:
circuit = p.Circuit()
root = p.compile_from_cnf_using_ganakarjun(circuit, xor_path)
circuit.set_root(root)
circuit.remove_unused_nodes()

print(f"[Ganak+Arjun] XOR circuit with nodes: {circuit.nb_nodes()}, roots: {circuit.nb_root_nodes()}")

[Ganak+Arjun] XOR circuit with nodes: 15, roots: 1


### 2.3 SDD (Sentential Decision Diagrams)

You can compile via the SDD compiler directly from a CNF file, **or** pass an `SddNode` you already built using `pysdd`.

In [6]:
# 1) from a CNF file
circuit = p.Circuit()
root = p.compile_from_cnf_using_sdd(circuit, xor_path)
circuit.set_root(root)
circuit.remove_unused_nodes()
print(f"[SDD from file] XOR circuit with nodes: {circuit.nb_nodes()}")

# 2) from a pysdd SddNode object
from pysdd.sdd import SddManager

mgr, sdd_node = SddManager.from_cnf_file(xor_path.encode(), vtree_type=b"balanced")

circuit = p.Circuit()
root = p.compile_from_sdd(circuit, sdd_node)
circuit.set_root(root)
circuit.remove_unused_nodes()
print(f"[SDD from SddNode] XOR circuit with nodes: {circuit.nb_nodes()}")

[SDD from file] XOR circuit with nodes: 7
Read CNF: vars=3 clauses=2
[SDD from SddNode] XOR circuit with nodes: 7


### 2.4 d4v2

In [7]:
circuit = p.Circuit()
root = p.compile_from_cnf_using_d4v2(circuit, xor_path)
circuit.set_root(root)
circuit.remove_unused_nodes()
print(f"[d4v2] XOR circuit with nodes: {circuit.nb_nodes()}")

[d4v2] XOR circuit with nodes: 14


### 2.5 Side-by-side comparison across backends

In [8]:
backends = {
    "ganak":         p.compile_from_cnf_using_ganak,
    "ganak+arjun":   p.compile_from_cnf_using_ganakarjun,
    "sdd":           p.compile_from_cnf_using_sdd,
    "d4v2":          p.compile_from_cnf_using_d4v2,
}

formulas = {
    "XOR": xor_path,
    "UNSAT": unsat_path,
}

print(f"{'Formula':<15} {'Backend':<18} {'Nodes':>6} {'Roots':>6}")
print("-" * 50)

for fname, fpath in formulas.items():
    for bname, bfn in backends.items():
        circ = p.Circuit()
        nptr = bfn(circ, fpath)
        circ.set_root(nptr)
        circ.remove_unused_nodes()
        print(f"{fname:<15} {bname:<18} {circ.nb_nodes():>6} {circ.nb_root_nodes():>6}")

Formula         Backend             Nodes  Roots
--------------------------------------------------
XOR             ganak                  15      1
XOR             ganak+arjun            15      1
XOR             sdd                     7      1
XOR             d4v2                   14      1
UNSAT           ganak                   4      1
UNSAT           ganak+arjun             4      1
UNSAT           sdd                     1      1
UNSAT           d4v2                    1      1


---
## 3. The `GatedFormula` API

Syntax:

1. `gf.add_input(name)`
2. `gf.add_and(gate_name, [children])`
3. `gf.add_or(gate_name, [children])`
4. `gf.add_target(gate_name)`

Then compile with `compile_from_gates_formula_using_d4v2(circuit, gf)`.

In [9]:
# x1 AND x2
gf = p.GatedFormula()
x1 = gf.add_input("x1")
x2 = gf.add_input("x2")
g_and = gf.add_and("g_and", [x1, x2])
gf.add_target(g_and)

circuit = p.Circuit()
root = p.compile_from_gates_formula_using_d4v2(circuit, gf)
circuit.set_root(root)
circuit.remove_unused_nodes()
print(f"AND(x1,x2) with nodes: {circuit.nb_nodes()}")

# x1 OR -x2
gf2 = p.GatedFormula()
gf2.add_input("x1")
gf2.add_input("x2")
gf2.add_or("g_or", ["x1", "-x2"])
gf2.add_target("g_or")

circuit2 = p.Circuit()
root2 = p.compile_from_gates_formula_using_d4v2(circuit2, gf2)
circuit2.set_root(root2)
circuit2.remove_unused_nodes()
print(f"OR(x1,-x2) with nodes: {circuit2.nb_nodes()}")

AND(x1,x2) with nodes: 4
OR(x1,-x2) with nodes: 15


In [10]:
# helper: encode any CNF as a GatedFormula
def cnf_to_gated_formula(n_vars: int, clauses: List[List[int]]) -> p.GatedFormula:
    """Encode a CNF as an equivalent GatedFormula."""
    gf = p.GatedFormula()
    for v in range(1, n_vars + 1):
        gf.add_input(str(v))

    next_id = n_vars + 1

    # no clauses → OR(x,-x) for every x
    if not clauses:
        taut_ids = []
        for v in range(1, n_vars + 1):
            gf.add_or(str(next_id), [str(v), str(-v)])
            taut_ids.append(str(next_id))
            next_id += 1
        gf.add_or(str(next_id), taut_ids)
        gf.add_target(str(next_id))
        return gf

    clause_gate_ids = []
    for clause in clauses:
        gf.add_or(str(next_id), [str(lit) for lit in clause])
        clause_gate_ids.append(str(next_id))
        next_id += 1

    if len(clause_gate_ids) == 1:
        gf.add_target(clause_gate_ids[0])
    else:
        gf.add_and(str(next_id), clause_gate_ids)
        gf.add_target(str(next_id))

    return gf


gf_xor = cnf_to_gated_formula(n_vars=3, clauses=[[1, 2], [-1, -2]])
circ_gf = p.Circuit()
root_gf = p.compile_from_gates_formula_using_d4v2(circ_gf, gf_xor)
circ_gf.set_root(root_gf)
circ_gf.remove_unused_nodes()
print(f"XOR via GatedFormula with nodes: {circ_gf.nb_nodes()}")

XOR via GatedFormula with nodes: 14


---
## 4. The `.bc` (Boolean Circuit) File Format

The `.bc` format describes a circuit gate by gate:

```
c  Comment line
I <var_name>                  # input (identity gate)
G <out> := A <in1> <in2> ...  # AND gate
G <out> := O <in1> <in2> ...  # OR  gate
T <output_literal>            # Target literal
```

### Example `.bc` file (`circ1.bc`)

```
c BC-S1.2
T root
I i1
I i2
I i0
G g0 := A i1 i2 i0
G g1 := O i1 -g0 -i2
G g2 := A -g0 i0 i1 i2
G g3 := O -g1 i2
G g4 := O -g1 -g2 -i2
G root := O g4 g3
```

In [11]:
import os

base_dir = os.path.dirname(os.path.abspath("."))
bc_path = "../assets/circuits/circ1.bc"

with open(bc_path) as f:
    print(f.read())

c BC-S1.2
T root
I i1
I i2
I i0
G g0 := A i1 i2 i0
G g1 := O i1 -g0 -i2
G g2 := A -g0 i0 i1 i2
G g3 := O -g1 i2
G g4 := O -g1 -g2 -i2
G root := O g4 g3



In [12]:
# compile directly from a .bc file
circuit_bc = p.Circuit()
root_bc = p.compile_from_gates_file_using_d4v2(circuit_bc, bc_path)
circuit_bc.set_root(root_bc)
circuit_bc.remove_unused_nodes()
print(f"circ1.bc with nodes: {circuit_bc.nb_nodes()}, roots: {circuit_bc.nb_root_nodes()}")

circ1.bc with nodes: 31, roots: 1


---
## 5. Combining Circuits: `or_node` and the Shared Circuit

A single `Circuit` object can host **multiple compiled formulas** and combine their roots with high-level operators.

> **Important:** call `set_root(nptr)` after **each** `compile_from_*` call before invoking the next one.

### 5.1 Sequential compilation and root accumulation

In [13]:
toy_path = "../assets/toy"
toy0_path = os.path.join(toy_path, "toy0.cnf")
toy1_path = os.path.join(toy_path, "toy1.cnf")

circuit = p.Circuit()

# compile first formula
nptr1 = p.compile_from_cnf_using_ganak(circuit, toy0_path)
circuit.set_root(nptr1)
nb_after_first = circuit.nb_nodes()
print(f"After first formula with nodes: {nb_after_first}, roots: {circuit.nb_root_nodes()}")

# compile second formula into the SAME circuit
nptr2 = p.compile_from_cnf_using_ganak(circuit, toy1_path)
circuit.set_root(nptr2)
nb_after_second = circuit.nb_nodes()
print(f"After second formula with nodes: {nb_after_second}, roots: {circuit.nb_root_nodes()}")

# structural sub-circuits are shared
print(f"Extra nodes from second formula: {nb_after_second - nb_after_first}")

After first formula with nodes: 170, roots: 1
After second formula with nodes: 237, roots: 2
Extra nodes from second formula: 67


### 5.2 Creating an OR node over existing roots

In [14]:
# build a disjunction: F1 ∨ F2
nptr3 = circuit.or_node([nptr1, nptr2])
circuit.set_root(nptr3)

print(f"After or_node with nodes: {circuit.nb_nodes()}, roots: {circuit.nb_root_nodes()}")

After or_node with nodes: 251, roots: 3


### 5.3 Garbage collection with `remove_unused_nodes`

In [15]:
# without setting nptr3 as root, it is unreachable from nptr1, nptr2
# remove_unused_nodes prunes everything not reachable from a root

circuit_gc = p.Circuit()
n1 = p.compile_from_cnf_using_ganak(circuit_gc, toy0_path)
n2 = p.compile_from_cnf_using_ganak(circuit_gc, toy1_path)
circuit_gc.or_node([n1, n2])

before = circuit_gc.nb_nodes()
circuit_gc.remove_unused_nodes()
after = circuit_gc.nb_nodes()

print(f"Nodes before pruning: {before}")
print(f"Nodes after pruning: {after}")

Nodes before pruning: 251
Nodes after pruning: 14


---
## 6 API Quick Reference

### Compile functions

```python
import kompyle as p

circuit = p.Circuit()

# from a .cnf file
nptr = p.compile_from_cnf_using_ganak(circuit, "/path/to/file.cnf")
nptr = p.compile_from_cnf_using_ganakarjun(circuit, "/path/to/file.cnf")
nptr = p.compile_from_cnf_using_sdd(circuit, "/path/to/file.cnf")
nptr = p.compile_from_cnf_using_d4v2(circuit, "/path/to/file.cnf")

# from a pysdd SddNode
nptr = p.compile_from_sdd(circuit, sdd_node)

# from a GatedFormula object
gf = p.GatedFormula()
gf.add_input("x1") 
gf.add_input("x2")
gf.add_and("g", ["x1", "x2"])
gf.add_target("g")
nptr = p.compile_from_gates_formula_using_d4v2(circuit, gf)

# from a .bc file
nptr = p.compile_from_gates_file_using_d4v2(circuit, "/path/to/file.bc")
```